# EDA учебной базы данных afisha

**Задача.** Предварительный анализ базы `data-analyst-afisha` (5 таблиц) по полному заданию:

1. **Взаимосвязи таблиц** — первичные (PK) и внешние (FK) ключи, типы связей (1:1, 1:M, M:N).
2. **Содержимое таблиц** — соответствие описанию, объём данных.
3. **Корректность данных** — уникальность идентификаторов, пропуски, написание категориальных значений (устройства, города, регионы, валюты).
4. **Распределение заказов по категориям** — равномерность, доминирующие и малочисленные категории.
5. **Аномалии в выручке** — статистика, выбросы, особенности.
6. **Период и сезонность** — за какой период данные, возможны ли сезонные эффекты.

Основной фокус — таблицы `purchases` и `events`: активность клиентов, типы мероприятий, выручка сервиса.

**Инструменты:** PostgreSQL через DBeaver (основной), pandas (контрольная проверка, TODO).

**Как читать ноутбук.** Каждый этап — связка: *что проверяем и зачем → как (запрос) → что получили → вывод*. Код вторичен: результаты зафиксированы в тексте, ноутбук можно прочитать, не выполняя ни одной ячейки.

**Карта задания → этапы:**

| # | Этап | Пункт задания | Статус |
|---|---|---|---|
| 1 | Схема: ключи и типы связей | Взаимосвязи таблиц | ✅ |
| 2 | Проверка ключей самими данными | Взаимосвязи таблиц | ✅ |
| 3 | Объём данных и период | Содержимое + период | ✅ |
| 4 | Пропуски (NULL) | Корректность | ✅ |
| 5 | Категориальные значения | Корректность | ✅ |
| 6 | Уникальность идентификаторов и дубликаты | Корректность | ✅ |
| 7 | Расследование аномалии revenue = 0 | Аномалии | ✅ |
| 8 | Отрицательная выручка | Аномалии | ✅ |
| 9 | Распределение заказов по категориям | Распределение | ⬜ TODO |
| 10 | Статистика выручки и выбросы | Аномалии, выручка | ⬜ TODO |
| 11 | Сезонность активности | Период, сезонные факторы | ⬜ TODO |
| 12 | Сводные выводы | — | ✅ / дополняется |
| 13 | Открытые проверки и контроль в pandas | Все пункты | ⬜ TODO |


## Этап 1. Схема: ключи и типы связей

**Что и зачем.** Прежде чем считать метрики, нужно понять устройство базы: какая таблица на какую ссылается и по каким полям. Это определяет, как писать JOIN и по чему группировать.

По описанию схемы вырисовывается такая картина ключей:

| Таблица | PK (первичный ключ) | FK (внешние ключи) |
|---|---|---|
| purchases | order_id | event_id → events |
| events | event_id | city_id → city, venue_id → venues |
| city | city_id | region_id → regions |
| venues | venue_id | — |
| regions | region_id | — |

Типы связей: **все связи «один ко многим» (1:M)**. Связей «один к одному» нет (ни одна таблица не расширяет другую строка-в-строку), «многие ко многим» тоже нет (нет и промежуточных таблиц). FK всегда лежит на стороне «многих».

```
regions —< city —< events —< purchases
venues  —< events
```

**Вывод этапа.** Схема — классическая «звезда» справочников вокруг таблицы фактов `purchases`. Для аналитики это значит: основная таблица — `purchases`, измерения подтягиваются JOIN'ами через `events`. Схема, впрочем, — лишь заявлено в описании; следующий этап проверяет её реальными данными.

## Этап 2. Проверка ключей самими данными

**Что и зачем.** Описание схемы может расходиться с реальностью. Тип ключа проверяется простым приёмом: сравнением `COUNT(*)` и `COUNT(DISTINCT колонка)` внутри конкретной таблицы:
- числа равны → колонка уникальна в этой таблице → ведёт себя как PK;
- `DISTINCT` заметно меньше → значения повторяются → колонка здесь FK, и связь с родительской таблицей 1:M.

**Проверка 1: `order_id` в purchases — ключ?**

In [ ]:
SELECT COUNT(*)               AS total_rows,
       COUNT(order_id)        AS non_null_ids,
       COUNT(DISTINCT order_id) AS unique_ids
FROM afisha.purchases;

**Результат:** 292 034 = 292 034 = 292 034 — три числа совпадают.

**Проверка 2: `event_id` в purchases — уникален или повторяется?**

In [ ]:
SELECT COUNT(*)                  AS total_rows,
       COUNT(DISTINCT event_id) AS unique_events
FROM afisha.purchases;

**Результат:** 292 034 строки, но только 22 484 уникальных `event_id` → в среднем ≈ 13 покупок на мероприятие.

**Вывод этапа.** Заявленная схема подтвердилась данными:
- `order_id` — полноценный PK таблицы `purchases`: один заказ = одна строка (это важно: покупки не «разбиты» на позиции, `SUM` по заказу не нужен);
- `event_id` в purchases — FK: связь `events` → `purchases` действительно «один ко многим», ≈13 заказов на мероприятие.

Метод на будущее: пара `COUNT(*)` vs `COUNT(DISTINCT)` — самый быстрый способ проверить, является ли поле ключом таблицы.

## Этап 3. Объём данных и период

**Что и зачем.** Пункт «содержимое таблиц» требует понять, в каком объёме представлены данные и за какой период. От охвата зависит, какие выводы вообще правомерны: 5 месяцев — это уже сезонность, но ещё не годовой цикл.

**Объём таблиц:**

| Таблица | Строк |
|---|---|
| purchases (заказы) | 292 034 |
| events (мероприятия) | 22 484 |
| city (города) | 353 |
| regions (регионы) | 81 |
| уникальные пользователи в purchases | 22 000 |

In [ ]:
SELECT MIN(created_dt_msk),
       MAX(created_dt_msk),
       COUNT(DISTINCT created_dt_msk),
       MAX(created_dt_msk) - MIN(created_dt_msk)
FROM afisha.purchases;

**Результат:** с 2024-06-01 по 2024-10-31, 153 уникальные даты при интервале 152 дня (152 + 1, обе границы включены).

**Вывод этапа.** Данные покрывают 5 месяцев (июнь–октябрь 2024), причём **без единого пропуска по дням** — есть заказы за каждые сутки периода. Объём солидный: ~2 тыс. заказов в день, 22 тыс. пользователей, география — 81 регион.

Ограничение для интерпретаций: сезон летних фестивалей и начала театрального сезона; выводы о годовом цикле на этих данных делать нельзя.

## Этап 4. Пропуски (NULL)

**Что и зачем.** Пропуски определяют, каким агрегатам можно верить: `COUNT(колонка)` игнорирует NULL, и «средний чек», посчитанный по колонке с пропусками, окажется искажён. Проверять надо каждую колонку каждой таблицы — это 30 проверок.

**Как автоматизировано.** Писать 30 запросов руками — не наш метод. Один запрос-генератор собирает их автоматически из служебной таблицы `information_schema.columns`, где СУБД хранит список всех колонок:

In [ ]:
-- Шаг 1: генератор. Собирает единую проверку по всем колонкам всех таблиц
SELECT string_agg(
    format(
        'SELECT %I AS table_name, %L AS column_name, COUNT(*) - COUNT(%I) AS null_count FROM %I.%I',
        table_name, column_name, column_name, table_schema, table_name
    ),
    ' UNION ALL '
)
FROM information_schema.columns
WHERE table_schema = 'afisha'
  AND table_name IN ('purchases', 'events', 'venues', 'city', 'regions');

-- Шаг 2: выполнить текст из ячейки результата — одна большая UNION-проверка

**Результат:** все 30 проверок вернули `null_count = 0` — ни одного пропуска ни в одной колонке всех пяти таблиц.

**Вывод этапа.** База без пропусков (типично для учебного набора, редкость для боевого). Практическое следствие: `COUNT(*)` и `COUNT(колонка)` везде эквивалентны, средние и суммы можно считать без оговорок про NULL. Приём с генерацией запросов из `information_schema` переносится на любой проект.

## Этап 5. Категориальные значения

**Что и зачем.** Пункт задания требует проверить «написание» категориальных данных: опечатки и нестандартные значения ломают GROUP BY (одна и та же сущность разъезжается на несколько групп). Метод: `GROUP BY` + `COUNT(*)` по каждой категориальной колонке и ревизия списка значений глазами.

**Результаты по всем категориальным полям:**

| Поле | Значений | Наблюдения |
|---|---|---|
| cinema_circuit | 6 | «нет» — 290 632 (99%+); сети: Другое 1 263, КиноСити 124, Киномакс 7, Москино 7, ЦентрФильм 1. Почти все мероприятия вне кинотеатров |
| currency_code | 2 | rub — 286 961; kzt — 5 073 |
| device_type_canonical | 5 | mobile — 232 679 (~80%); desktop — 58 170; tablet — 1 180; other — 2; tv — 3 |
| age_limit | 5 | только допустимые значения: 0 / 6 / 12 / 16 / 18 |
| service_name | 36 | 36 билетных операторов |

**Вывод этапа.** Все категориальные поля — в «каноническом» виде: опечаток, синонимов и мусорных значений нет.

Два наблюдения для будущей аналитики:
1. **Две валюты** — суммы `total`/`revenue` нельзя агрегировать в одну цифру без разбивки по валюте (тенге и рубли в одном SUM — бессмысленное число);
2. **Mobile-first** — 80% заказов с мобильных устройств: пользовательская аналитика должна начинаться с mobile.

Пример запроса-ревизии (остальные поля проверялись аналогично):

In [ ]:
SELECT device_type_canonical, COUNT(*)
FROM afisha.purchases
GROUP BY device_type_canonical
ORDER BY COUNT(*) DESC;

## Этап 6. Уникальность идентификаторов и дубликаты

**Что и зачем.** Дубликаты бывают двух видов: полные (одна строка скопирована) — грязь, и «частичные» (строки различаются лишь служебным id) — возможная бизнес-логика. Полных дублей по всем таблицам нет (`duplicated`-проверка). Но в `events` обнаружилось подозрительное: строк больше, чем уникальных комбинаций содержательных полей.

**Проверка:** группировка по всем колонкам, кроме `event_id`, через `ROW_NUMBER()`:

In [ ]:
SELECT COUNT(*) AS duplicate_rows
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY event_name_code, event_type_description,
                              event_type_main, organizers, city_id, venue_id
                              ORDER BY event_id) AS rn
    FROM afisha.events
) t
WHERE rn > 1;

**Результат:** 873 строки идентичны по всем колонкам, кроме `event_id`.

**Вывод этапа.** В справочнике `events` одно и то же мероприятие может существовать под несколькими `event_id` — вероятная причина: одинаковые мероприятия, проходившие в разные даты (или дубль при загрузке справочника). Практическое следствие для аналитики: **мероприятие нельзя считать единицей, равной `event_id`** — при подсчёте «сколько мероприятий» нужно группировать по содержательной комбинации полей.

Попутная находка: город **«Глинополье» встречается в `city` дважды — в разных регионах**. Это не грязь, а реальная коллизия имён. Следствие: группировать по городу только по `city_id`, но не по `city_name`.

## Этап 7. Расследование аномалии: 5 772 заказа с revenue = 0

**Что и зачем.** ≈2% заказов имеют нулевую выручку. Это либо особенность бизнеса (бесплатные события), либо дефект данных (потерянные суммы, отмены). Различить важно: от этого зависит, можно ли доверять метрикам выручки.

Это самый интересный пункт исследования — цепочка из четырёх шагов, где каждый следующий уточняет гипотезу, оставшуюся после предыдущего.

### Шаг 1. Есть ли у zero-заказов билеты?

**Гипотеза:** если билеты не выписаны (`tickets_count = 0`) — это «пустые»/отменённые заказы; если выписаны — заказ живой, и ноль выручки означает что-то другое.

In [ ]:
SELECT tickets_count, COUNT(*)
FROM afisha.purchases
WHERE revenue = 0
GROUP BY tickets_count
ORDER BY tickets_count;

**Результат:** все 5 772 заказа имеют `tickets_count ≥ 1` (в основном 1–6 билетов; редкие выбросы до 57).

**Вывод шага:** заказы «живые» — билеты есть. Гипотеза «отменённые пустые заказы» исключена. Остаются: бесплатные мероприятия, промо, возвраты.

### Шаг 2. Нулевая выручка — свойство заказа или мероприятия?

**Гипотеза:** если zero-заказы размазаны по мероприятиям, у которых остальная выручка есть, — это возвраты/промо на уровне заказов. Если же нулевая выручка у мероприятий **целиком** — это бесплатные события. Проверяем группировкой по `event_id`.

In [ ]:
SELECT
    COUNT(*) AS zero_events,
    SUM(flag_0) AS events_with_mixed_revenue
FROM (
    SELECT p.event_id,
           MAX(CASE WHEN p.revenue > 0 THEN 1 ELSE 0 END) AS flag_0
    FROM afisha.purchases p
    GROUP BY p.event_id
    HAVING SUM(p.revenue) = 0
) t;

**Результат:** **295 мероприятий** с нулевой суммарной выручкой, и **ни у одного из них нет ни одного платного заказа** (`events_with_mixed_revenue = 0`).

**Вывод шага:** нулевая выручка — свойство **мероприятия целиком**, а не отдельных заказов. «Смешанных» мероприятий нет. Гипотеза возвратов на уровне заказов практически исключена; остаётся главная — бесплатные мероприятия.

Приём этапа: `MAX(CASE WHEN условие THEN 1 ELSE 0 END)` — «условие выполнилось хотя бы раз в группе». Он страхует от ложной интерпретации: нулевая сумма может получиться и взаимным погашением плюс-минус (платёж + возврат). Именно поэтому шаг 3 обязателен.

### Шаг 3. Может ли возврат «погасить» мероприятие до нуля?

**Гипотеза:** если в базе есть крупные отрицательные выручки (полные возвраты), то мероприятие из шага 2 может быть платным со стопроцентным возвратом. Проверяем минимум выручки по всей таблице.

In [ ]:
SELECT MIN(revenue), MIN(total)
FROM afisha.purchases;

**Результат:** минимальная выручка — **−90**, минусы встречаются, но они копеечные (в основном несколько рублей).

**Вывод шага:** погасить платный заказ (сотни+ рублей) минусом в −90 невозможно: `+500 − 90 = +410 ≠ 0`. Значит, zero-мероприятия из шага 2 — **не** скрытые возвраты. Математически исключаем последнюю альтернативную гипотезу.

### Шаг 4. Контроль на выборке

Точечная проверка нескольких zero-мероприятий (`event_id` 533566, 553623, 548089 и др.): у всех заказов этих мероприятий `revenue = 0` и `total = 0` — нулевые суммы по всей длине цепочки, включая сумму, заплаченную клиентом.

**Итог этапа.** Гипотеза доказана цепочкой: **295 мероприятий базы — полностью бесплатные** (≈20 заказов на каждое, ≈2% всех заказов). Возвраты и отмены исключены по данным. Отдельного статуса оплаты в схеме нет, поэтому оставшиеся сомнения (промо vs. бесплатные) средствами этой базы неразрешимы — это честная граница исследования.

## Этап 8. Отрицательная выручка

**Что и зачем.** На шаге 3 предыдущего этапа обнаружено: 381 заказ имеет `revenue < 0` (мин. −90, обычно несколько рублей).

**Гипотеза:** копеечные минуса — это комиссия платёжного шлюза, удержания при возврате сервисного сбора или корректировки биллинга — то есть «технические» суммы, а не возвраты билетов (возврат дал бы минус в размере целой стоимости заказа).

**Значение для исследования:** малость минусов — ключевое условие вывода этапа 7 (минус не может погасить платёж до нуля).

**Открытый вопрос:** точная природа этих 381 заказа. «Паспорт» смотрится запросом ниже — если `revenue` маленький, а `total` нормальный, это подтверждает биллинговую природу; если минус у одного конкретного `service_name` — особенность оператора.

In [ ]:
SELECT p.revenue, p.total, p.tickets_count, p.currency_code, p.service_name
FROM afisha.purchases p
WHERE p.revenue < 0
ORDER BY p.revenue
LIMIT 20;

## Этап 9. Распределение заказов по категориям ⬜ TODO

**Что и зачем (пункт 4 задания).** Проверить, равномерно ли распределены заказы по категориям: какие типы мероприятий значительно популярнее остальных, с каких устройств чаще покупают, какой валютой платят. Зачем это EDA: перекос распределения определяет, где метрики надёжны (многочисленные категории), а где нестабильны (малочисленные); редкие категории — кандидаты на объединение или отдельную проверку.

**Что уже известно из предыдущих этапов (и уже говорит о неравномерности):**

| Признак | Крупнейшая категория | Малые категории (⚠️) |
|---|---|---|
| device_type_canonical | mobile — ~80% заказов | other — 2 заказа, tv — 3 |
| currency_code | rub — 98.3% | kzt — 1.7% (5 073) |
| cinema_circuit | нет — 99.6% | ЦентрФильм — 1 заказ |
| age_limit | относительно ровно: 16+ (78 864), 12+ (62 861), 0+ (61 731) | — |

**Что осталось прогнать (результаты TODO):**

1. Главный разрез — **тип мероприятия** (`event_type_main`): заказы, доли и выручка:

In [ ]:
SELECT e.event_type_main,
       COUNT(*) AS orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS share_pct,
       SUM(p.total) FILTER (WHERE p.currency_code = 'rub') AS revenue_rub
FROM afisha.purchases p
JOIN afisha.events e ON p.event_id = e.event_id
GROUP BY e.event_type_main
ORDER BY orders DESC;

2. География: заказы по городам и регионам (топ и «хвост»):

```sql
-- TODO: аналогично GROUP BY city_name (через JOIN city) и region_name,
-- с долей через оконное SUM(COUNT(*)) OVER ()
```

**Вопросы для интерпретации (ответы TODO):**
- Какие `event_type_main` дают >10% заказов? Одна доминирующая категория или несколько сопоставимых?
- Есть ли типы мероприятий с большим числом заказов, но малой выручкой (например, за счёт 295 бесплатных)?
- Насколько длинный «хвост» малых категорий — стоит ли объединять их в «Прочие» при визуализации?

**Вывод этапа:** TODO после прогона.

## Этап 10. Статистика выручки и выбросы ⬜ TODO

**Что и зачем (пункт 5 задания).** Изучить статистику выручки: типичные значения, разброс, выбросы. Аномалии, уже найденные и объяснённые: нулевая выручка (295 бесплатных мероприятий, этап 7) и копеечная отрицательная (381 заказ, этап 8). Здесь — общий профиль распределения.

**Что осталось прогнать:**

1. Ключевые статистики `total` и `revenue` (только rub, чтобы масштабы не смешивались; только `revenue > 0`, чтобы бесплатные и минуса не тянули среднее вниз):

In [ ]:
SELECT
    MIN(total)    AS min_total,
    MAX(total)    AS max_total,
    ROUND(AVG(total), 2) AS mean_total,
    PERCENTILE_CONT(0.5)  WITHIN GROUP (ORDER BY total) AS median_total,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY total) AS p95_total
FROM afisha.purchases
WHERE currency_code = 'rub' AND revenue > 0;

2. Выбросы по межквартильному размаху (IQR):

In [ ]:
WITH stats AS (
    SELECT PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY total) AS q1,
           PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY total) AS q3
    FROM afisha.purchases
    WHERE currency_code = 'rub' AND revenue > 0
)
SELECT COUNT(*) AS outliers,
       MIN(p.total) AS min_outlier,
       MAX(p.total) AS max_outlier
FROM afisha.purchases p, stats s
WHERE p.currency_code = 'rub' AND p.revenue > 0
  AND (p.total < s.q1 - 1.5 * (s.q3 - s.q1)
    OR p.total > s.q3 + 1.5 * (s.q3 - s.q1));

3. Связь `total` и `revenue`: гипотеза — `revenue` это выручка сервиса (доля от `total`). Проверка: `revenue / total` — постоянная ли доля? 4. Топ-10 заказов по `total` — что это за мероприятия, разовая крупная покупка или ошибка.

**Вопросы для интерпретации (ответы TODO):**
- Среднее vs медиана: если среднее заметно выше медианы — распределение скошено вправо «хвостом» крупных заказов.
- Каков разброс выбросов: ожидаемо, что хвост тяжёлый (билетные покупки варьируются от 1 билета до групп на 50+).
- Доля `revenue` от `total` — комиссия сервиса или что-то иное?

**Вывод этапа:** TODO после прогона.

## Этап 11. Сезонность активности ⬜ TODO

**Что и зачем (пункт 6 задания).** Период установлен (этап 3): 2024-06-01 → 2024-10-31, 153 дня без пропусков. Теперь — динамика внутри периода и влияние сезонных факторов.

**Что осталось прогнать:**

1. Динамика по месяцам (заказы и выручка):

In [ ]:
SELECT DATE_TRUNC('month', created_dt_msk) AS month,
       COUNT(*) AS orders,
       SUM(total) FILTER (WHERE currency_code = 'rub') AS revenue_rub
FROM afisha.purchases
GROUP BY 1
ORDER BY 1;

2. Профиль по дням недели (`EXTRACT(DOW FROM created_dt_msk)`) и по часам суток (`EXTRACT(HOUR FROM created_ts_msk)`) — аналогичными GROUP BY.

**Вопросы для интерпретации (ответы TODO):**
- Растёт ли активность к осени (начало театрального сезона) или падает (конец летних каникул)?
- Есть ли всплески по дням недели (пятница/выходные — логичная гипотеза для билетов) и по часам (вечер vs. рабочий день)?
- Важно: 5 месяцев — один тёплый сезон. **Годовой цикл на этих данных не виден**: сравнить июнь с декабрём нельзя, и любые «сезонные» выводы формулировать только внутри периода.

**Вывод этапа:** TODO после прогона.

## Этап 12. Сводные выводы

**Доказано (этапы 1–8):**

1. **Схема подтверждена данными.** Пять таблиц, все связи «один ко многим»; `order_id` уникален (292 034 = 292 034), `event_id` повторяется в purchases (≈13 заказов на мероприятие). Классическая звезда: факты в `purchases`, измерения в справочниках.
2. **Пропусков нет** — во всех 30 колонках пяти таблиц `null_count = 0`. Проверка автоматизирована через `information_schema`.
3. **Категориальные данные чистые** — канонические значения без опечаток. Особенности: 2 валюты (rub 98.3% / kzt 1.7%), mobile ~80% заказов, 36 операторов.
4. **Период:** июнь–октябрь 2024, 153 дня без пропусков, ~2 000 заказов/день, 22 000 пользователей, 81 регион.
5. **Главная аномалия объяснена:** 295 мероприятий полностью бесплатные (≈2% заказов) — доказано трёхшаговой проверкой (билеты есть → нет смешанных мероприятий → минусы малы для погашения).
6. **Артефакты для дальнейшей аналитики:** смешение валют (суммы только в разрезе `currency_code`); `event_id` ≠ мероприятие (873 дубля в справочнике); города-омонимы — группировать по `city_id`; 381 заказ с копеечной отрицательной выручкой (макс. −90).

**Дополнить после этапов 9–11 (TODO):** выводы о распределении по типам мероприятий, профиле выручки (среднее vs медиана, выбросы) и внутриperiod-ной динамике (месяцы, дни недели, часы).

## Этап 13. Открытые проверки (TODO)

**1. Referential integrity** — все ли FK указывают на существующие записи (ожидается 0; прогнать для всех четырёх связей):

In [ ]:
SELECT COUNT(*) AS broken_refs
FROM afisha.purchases p
LEFT JOIN afisha.events e ON p.event_id = e.event_id
WHERE e.event_id IS NULL;
-- TODO: аналогично events.city_id → city, events.venue_id → venues, city.region_id → regions

**2. Профиль бесплатных мероприятий** — разбить 295 zero-мероприятий по `event_type_main`: если бесплатные сконцентрированы в одном типе, это готовая бизнес-находка:

In [ ]:
SELECT e.event_type_main,
       COUNT(*) AS free_events,
       SUM(cnt)  AS free_orders
FROM (
    SELECT p.event_id, COUNT(*) AS cnt
    FROM afisha.purchases p
    GROUP BY p.event_id
    HAVING SUM(p.revenue) = 0
) t
JOIN afisha.events e ON t.event_id = e.event_id
GROUP BY e.event_type_main
ORDER BY free_orders DESC;

**3. Природа 381 заказа с отрицательной выручкой** — «паспорт» минусовых заказов (этап 8): связь `revenue` и `total`, зависимость от `service_name`.

**4. Контрольная проверка в pandas** — подтвердить SQL-выводы вторым инструментом; заодно визуализировать выручку (этап 10) и динамику (этап 11):

In [ ]:
# pip install sqlalchemy psycopg2-binary pandas matplotlib
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('postgresql+psycopg2://<user>:<password>@<host>:5432/<database>')
df = pd.read_sql('SELECT * FROM afisha.purchases', engine)

df.duplicated().sum()        # полные дубли       → ожидается 0
df.isna().sum()               # пропуски           → ожидается все нули
df['order_id'].is_unique      # уникальность ключа → ожидается True

rub = df[(df.currency_code == 'rub') & (df.revenue > 0)]
rub['total'].describe()                                   # статистики этапа 10
rub.boxplot(column='total')                                # выбросы глазами
df.groupby(df['created_dt_msk'].dt.month)['order_id'].count().plot()  # сезонность этапа 11